# Lab 8: Implementation and Performance Evaluation of Categorical Naive Bayes Classifier



In [4]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv('Lab 8.csv')
df.head()


,No,Outlook,Temperature,Humidity,Wind,Play Tennis
0,1,Sunny,Hot,High,Weak,No
1,2,Sunny,Hot,High,Strong,No
2,3,Overcast,Hot,High,Weak,Yes
3,4,Rain,Mild,High,Weak,Yes
4,5,Rain,Cool,Normal,Weak,Yes


In [5]:

X=df[['Outlook','Temperature','Humidity','Wind']].copy()
y=df['Play Tennis'].copy()

feature_encoders={}
for col in X.columns:
    le=LabelEncoder()
    X[col]=le.fit_transform(X[col])
    feature_encoders[col]=le

target_encoder=LabelEncoder()
y=target_encoder.fit_transform(y)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print("Training samples:",len(X_train))
print("Testing samples:",len(X_test))


Training samples: 40
Testing samples: 10


### Interpretation
All categorical values are converted into numerical form using Label Encoding. The dataset is split into 80% training data and 20% testing data.

In [6]:

nb=CategoricalNB()
nb.fit(X_train,y_train)
pred=nb.predict(X_test)

print("Naive Bayes Accuracy:",accuracy_score(y_test,pred))
print("\nConfusion Matrix")
print(confusion_matrix(y_test,pred))
print("\nClassification Report")
print(classification_report(y_test,pred,target_names=target_encoder.classes_))


Naive Bayes Accuracy: 0.9

Confusion Matrix
[[2 1]
 [0 7]]

Classification Report
              precision    recall  f1-score   support

          No       1.00      0.67      0.80         3
         Yes       0.88      1.00      0.93         7

    accuracy                           0.90        10
   macro avg       0.94      0.83      0.87        10
weighted avg       0.91      0.90      0.89        10



### Interpretation
The confusion matrix summarizes correct and incorrect predictions. Precision, Recall and F1-score provide a detailed evaluation of classification performance beyond simple accuracy.

In [7]:

sample={
'Outlook':'Sunny',
'Temperature':'Cool',
'Humidity':'High',
'Wind':'Strong'
}

sample_df=pd.DataFrame([sample])

for col in sample_df.columns:
    sample_df[col]=feature_encoders[col].transform(sample_df[col])

pred=nb.predict(sample_df)[0]
probs=nb.predict_proba(sample_df)[0]

print("Prediction:",target_encoder.inverse_transform([pred])[0])
print("Probabilities:")
for cls,p in zip(target_encoder.classes_,probs):
    print(f"{cls}: {p:.4f}")


Prediction: No
Probabilities:
No: 0.7894
Yes: 0.2106


### Interpretation
The trained Naive Bayes classifier predicts whether tennis will be played for the specified weather conditions and provides confidence values for each possible class.

In [8]:

models={
'Naive Bayes':CategoricalNB(),
'Decision Tree':DecisionTreeClassifier(random_state=42),
'Logistic Regression':LogisticRegression(max_iter=500),
'SVM':SVC(probability=True,random_state=42)
}

results=[]

for name,model in models.items():
    model.fit(X_train,y_train)
    test_acc=accuracy_score(y_test,model.predict(X_test))
    sample_pred=target_encoder.inverse_transform(model.predict(sample_df))[0]
    sample_prob=model.predict_proba(sample_df)[0]
    results.append({
        'Model':name,
        'Accuracy':round(test_acc,4),
        'Prediction':sample_pred,
        'Prob(No)':round(sample_prob[0],4),
        'Prob(Yes)':round(sample_prob[1],4)
    })

comparison=pd.DataFrame(results)
print(comparison)
comparison


                 Model  Accuracy Prediction  Prob(No)  Prob(Yes)
0          Naive Bayes       0.9         No    0.7894     0.2106
1        Decision Tree       1.0         No    1.0000     0.0000
2  Logistic Regression       0.9         No    0.8613     0.1387
3                  SVM       1.0         No    0.9569     0.0431


C:\Users\karun\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Prediction,Prob(No),Prob(Yes)
0,Naive Bayes,0.9,No,0.7894,0.2106
1,Decision Tree,1.0,No,1.0000,0.0000
2,Logistic Regression,0.9,No,0.8613,0.1387
3,SVM,1.0,No,0.9569,0.0431


### Analysis Report

The models produce different predictions and probability values because they use different learning principles. Categorical Naive Bayes assumes that all input features are conditionally independent, while Decision Trees learn decision rules by recursively splitting the data, Logistic Regression models a linear decision boundary, and SVM attempts to maximize the separation margin between classes. Since these algorithms learn different decision boundaries and probability estimation methods, their confidence scores and sometimes even their predicted classes may differ for the same weather conditions. Accuracy differences also arise because each algorithm generalizes differently from the limited training data.